# Data Cleaning Notebook - ProductLens AI

This notebook performs comprehensive data cleaning on the e-commerce dataset:
1. Remove special character noise
2. Handle missing values
3. Remove duplicate entries
4. Standardize data formats
5. Save cleaned dataset for vectorization

## 1. Environment Setup

In [84]:
# =============================================================================
# CELL 1: ENVIRONMENT SETUP
# =============================================================================

import pandas as pd
import numpy as np
import re
import sys
import warnings
from pathlib import Path
from typing import Optional

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

# =============================================================================
# PATH SETUP
# =============================================================================

def setup_backend_path(relative_path: str = '../backend') -> Path:
    """
    Add backend directory to Python path for importing services.
    
    Args:
        relative_path: Relative path to backend directory
        
    Returns:
        Resolved absolute path to backend directory
        
    Raises:
        FileNotFoundError: If backend directory doesn't exist
    """
    backend_path = Path(relative_path).resolve()
    
    if not backend_path.exists():
        raise FileNotFoundError(f"Backend directory not found: {backend_path}")
    
    if not backend_path.is_dir():
        raise NotADirectoryError(f"Backend path is not a directory: {backend_path}")
    
    if str(backend_path) not in sys.path:
        sys.path.insert(0, str(backend_path))
    
    return backend_path

# =============================================================================
# PANDAS DISPLAY CONFIGURATION
# =============================================================================

def configure_pandas_display(
    max_columns: Optional[int] = None,
    max_colwidth: int = 100,
    max_rows: int = 100,
    width: Optional[int] = None,
    precision: int = 2
) -> None:
    """Configure pandas display options for better notebook output."""
    pd.set_option('display.max_columns', max_columns)
    pd.set_option('display.max_colwidth', max_colwidth)
    pd.set_option('display.max_rows', max_rows)
    pd.set_option('display.width', width)
    pd.set_option('display.precision', precision)
    pd.set_option('display.float_format', lambda x: f'{x:,.{precision}f}')

# =============================================================================
# ENVIRONMENT VALIDATION
# =============================================================================

def validate_environment() -> dict:
    """Validate and return environment information."""
    env_info = {
        'python_version': sys.version.split()[0],
        'pandas_version': pd.__version__,
        'numpy_version': np.__version__,
        'platform': sys.platform,
    }
    
    min_versions = {'pandas': '1.5.0', 'numpy': '1.20.0'}
    
    for lib, min_ver in min_versions.items():
        current_ver = env_info[f'{lib}_version']
        if tuple(map(int, current_ver.split('.')[:2])) < tuple(map(int, min_ver.split('.')[:2])):
            warnings.warn(f"{lib} version {current_ver} is below recommended {min_ver}")
    
    return env_info

# =============================================================================
# INITIALIZATION
# =============================================================================

backend_path = setup_backend_path('../backend')
configure_pandas_display()
env_info = validate_environment()

print("=" * 60)
print("ENVIRONMENT SETUP COMPLETE")
print("=" * 60)
print(f"Backend path:    {backend_path}")
print(f"Python version:  {env_info['python_version']}")
print(f"Pandas version:  {env_info['pandas_version']}")
print(f"NumPy version:   {env_info['numpy_version']}")
print(f"Platform:        {env_info['platform']}")
print("=" * 60)

ENVIRONMENT SETUP COMPLETE
Backend path:    /home/chris/coding-task/productlens-ai/backend
Python version:  3.12.3
Pandas version:  2.1.3
NumPy version:   1.26.2
Platform:        linux


## 2. Load Raw Dataset

In [85]:
# =============================================================================
# CELL 2: LOAD DATA
# =============================================================================

def load_dataset(
    file_path: Path,
    encodings: list[str] = ['utf-8', 'latin-1', 'iso-8859-1', 'cp1252', 'utf-16'],
    sample_size: Optional[int] = None,
    low_memory: bool = False
) -> tuple[pd.DataFrame, str]:
    """
    Load CSV dataset with robust encoding detection and error handling.
    
    Args:
        file_path: Path to the CSV file
        encodings: List of encodings to try in order
        sample_size: If set, only load this many rows
        low_memory: If True, use chunked loading for large files
        
    Returns:
        Tuple of (DataFrame, encoding_used)
    """
    if not file_path.exists():
        raise FileNotFoundError(f"Dataset not found: {file_path}")
    
    if not file_path.is_file():
        raise ValueError(f"Path is not a file: {file_path}")
    
    file_size_mb = file_path.stat().st_size / (1024 * 1024)
    print(f"File size: {file_size_mb:.2f} MB")
    
    if file_size_mb > 500 and not low_memory:
        warnings.warn(f"Large file detected ({file_size_mb:.0f}MB). Consider using low_memory=True")
    
    df = None
    successful_encoding = None
    errors_encountered = {}
    
    for encoding in encodings:
        try:
            df = pd.read_csv(
                file_path,
                encoding=encoding,
                nrows=sample_size,
                low_memory=low_memory,
                on_bad_lines='warn',
                dtype_backend='numpy_nullable'
            )
            successful_encoding = encoding
            break
        except UnicodeDecodeError as e:
            errors_encountered[encoding] = str(e)
            continue
        except Exception as e:
            errors_encountered[encoding] = str(e)
            continue
    
    if df is None:
        error_summary = "\n".join([f"  {enc}: {err}" for enc, err in errors_encountered.items()])
        raise ValueError(f"Failed to load file with any encoding:\n{error_summary}")
    
    print(f"Successfully loaded with encoding: {successful_encoding}")
    return df, successful_encoding


def print_dataframe_summary(df: pd.DataFrame, name: str = "DataFrame") -> None:
    """Print a formatted summary of the DataFrame."""
    print("=" * 60)
    print(f"{name.upper()} SUMMARY")
    print("=" * 60)
    print(f"Rows:              {len(df):,}")
    print(f"Columns:           {len(df.columns)}")
    print(f"Memory usage:      {df.memory_usage(deep=True).sum() / (1024 * 1024):.2f} MB")
    print(f"Duplicated rows:   {df.duplicated().sum():,}")
    print("-" * 60)
    print("COLUMNS & TYPES:")
    print("-" * 60)
    
    for col in df.columns:
        null_count = df[col].isnull().sum()
        null_pct = (null_count / len(df)) * 100
        print(f"  {col:<25} {str(df[col].dtype):<15} nulls: {null_count:>6} ({null_pct:>5.1f}%)")
    print("=" * 60)


# Load data
raw_data_path = backend_path / 'data' / 'raw' / 'dataset.csv'
df_raw, encoding_used = load_dataset(raw_data_path)
print_dataframe_summary(df_raw, "Raw Dataset")

print("\nFIRST 5 ROWS:")
display(df_raw.head()) if 'display' in dir() else print(df_raw.head())

print("\nLAST 5 ROWS:")
display(df_raw.tail()) if 'display' in dir() else print(df_raw.tail())

File size: 50.80 MB
Successfully loaded with encoding: utf-8
RAW DATASET SUMMARY
Rows:              541,909
Columns:           8
Memory usage:      265.31 MB
Duplicated rows:   91
------------------------------------------------------------
COLUMNS & TYPES:
------------------------------------------------------------
  InvoiceNo                 string          nulls:      0 (  0.0%)
  StockCode                 string          nulls:      0 (  0.0%)
  Description               string          nulls:   1025 (  0.2%)
  Quantity                  string          nulls:      0 (  0.0%)
  InvoiceDate               string          nulls:      0 (  0.0%)
  UnitPrice                 string          nulls:      0 (  0.0%)
  CustomerID                string          nulls: 108000 ( 19.9%)
  Country                   string          nulls:      0 (  0.0%)

FIRST 5 ROWS:
  InvoiceNo StockCode                           Description Quantity  \
0    536365    85123A    WHITE HANGING HEART T-LIGHT HOLDE

## 3. Data Preview & Anomaly Scan

In [86]:
# =============================================================================
# CELL 3: DATA PREVIEW & ANOMALY SCAN
# =============================================================================

def preview_dataframe(df: pd.DataFrame, n_rows: int = 10) -> None:
    """Display a comprehensive preview of the DataFrame with anomaly detection."""
    print("=" * 60)
    print(f"DATA PREVIEW (First {n_rows} rows)")
    print("=" * 60)
    
    display(df.head(n_rows)) if 'display' in dir() else print(df.head(n_rows))
    
    print("\n" + "=" * 60)
    print("QUICK ANOMALY SCAN")
    print("=" * 60)
    
    for col in df.columns:
        col_str = df[col].astype(str)
        
        anomalies = {
            'non_ascii': col_str.str.contains(r'[^\x00-\x7F]', regex=True, na=False).sum(),
            'XxY_pattern': col_str.str.contains(r'XxY', regex=True, na=False).sum(),
            'Ww_pattern': col_str.str.contains(r'Ww', regex=True, na=False).sum(),
            'leading_symbols': col_str.str.contains(r'^[\$\#\@\&\^]', regex=True, na=False).sum(),
        }
        
        total_anomalies = sum(anomalies.values())
        if total_anomalies > 0:
            print(f"\n  {col}:")
            for anomaly_type, count in anomalies.items():
                if count > 0:
                    print(f"    - {anomaly_type}: {count:,} occurrences")


preview_dataframe(df_raw, n_rows=10)

DATA PREVIEW (First 10 rows)
  InvoiceNo StockCode                           Description Quantity  \
0    536365    85123A    WHITE HANGING HEART T-LIGHT HOLDER        6   
1    536365     71053                   WHITE METAL LANTERN        6   
2    536365  ö84406B^        CREAM CUPID HEARTS COAT HANGER        8   
3    536365    84029G  $KNITTED UNION FLAG HOT WATER BOTTLE       6@   
4    536365    84029E       $RED WOOLLY HOTTIE WHITE HEART.       6@   
5    536365     22752          SET 7 BABUSHKA NESTING BOXES       2@   
6   536365ä   ö21730^    $GLASS STAR FROSTED T-LIGHT HOLDER       6@   
7   536366ä     22633                HAND WARMER UNION JACK        6   
8   536366ä   ö22632^             HAND WARMER RED POLKA DOT        6   
9   536367ä   ö84879^        $ASSORTED COLOUR BIRD ORNAMENT       32   

           InvoiceDate UnitPrice CustomerID              Country  
0  2010-12-01 08:26:00      2.55    17850.0  XxYUnited Kingdom☺️  
1  2010-12-01 08:26:00      3.39    17850.0 

## 4. Dataset Info & Statistics

In [87]:
# =============================================================================
# CELL 4: DATASET INFO & STATISTICS
# =============================================================================

def comprehensive_dataset_info(df: pd.DataFrame, name: str = "Dataset") -> None:
    """Generate and display comprehensive dataset information."""
    print("=" * 60)
    print(f"{name.upper()} - DETAILED INFO")
    print("=" * 60)
    
    # Memory usage per column
    memory_usage = df.memory_usage(deep=True)
    total_memory_mb = memory_usage.sum() / (1024 * 1024)
    
    print(f"\nMemory Usage: {total_memory_mb:.2f} MB")
    print("-" * 40)
    for col in df.columns:
        col_memory = memory_usage[col] / (1024 * 1024)
        print(f"  {col:<25} {col_memory:>8.2f} MB")
    
    # Data types
    print("\n" + "-" * 40)
    print("Data Types Summary:")
    print("-" * 40)
    for dtype, count in df.dtypes.value_counts().items():
        print(f"  {str(dtype):<20} {count} columns")
    
    # Pandas info
    print("\n" + "=" * 60)
    print("PANDAS INFO OUTPUT")
    print("=" * 60)
    df.info(memory_usage='deep')
    
    # Unique values
    print("\n" + "=" * 60)
    print("UNIQUE VALUES PER COLUMN")
    print("=" * 60)
    for col in df.columns:
        unique_count = df[col].nunique()
        unique_pct = (unique_count / len(df)) * 100
        print(f"  {col:<25} {unique_count:>10,} unique ({unique_pct:>6.2f}%)")


comprehensive_dataset_info(df_raw, "Raw Dataset")

RAW DATASET - DETAILED INFO

Memory Usage: 276.02 MB
----------------------------------------
  InvoiceNo                    35.17 MB
  StockCode                    34.96 MB
  Description                  39.22 MB
  Quantity                     26.21 MB
  InvoiceDate                  35.14 MB
  UnitPrice                    27.59 MB
  CustomerID                   29.04 MB
  Country                      48.67 MB

----------------------------------------
Data Types Summary:
----------------------------------------
  string               8 columns

PANDAS INFO OUTPUT
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   InvoiceNo    541909 non-null  string
 1   StockCode    541909 non-null  string
 2   Description  540884 non-null  string
 3   Quantity     541909 non-null  string
 4   InvoiceDate  541909 non-null  string
 5   UnitPrice    541909 n

## 5. Missing Values Analysis

In [88]:
# =============================================================================
# CELL 5: MISSING VALUES ANALYSIS
# =============================================================================

def analyze_missing_values(df: pd.DataFrame) -> pd.DataFrame:
    """Comprehensive missing value analysis."""
    print("=" * 60)
    print("MISSING VALUES ANALYSIS")
    print("=" * 60)
    
    missing_stats = pd.DataFrame({
        'Missing Count': df.isnull().sum(),
        'Missing %': (df.isnull().sum() / len(df) * 100).round(4),
        'Present Count': df.notnull().sum(),
        'Dtype': df.dtypes
    }).sort_values('Missing %', ascending=False)
    
    total_cells = df.shape[0] * df.shape[1]
    total_missing = df.isnull().sum().sum()
    
    print(f"\nOverall Statistics:")
    print(f"  Total cells:           {total_cells:,}")
    print(f"  Total missing:         {total_missing:,}")
    print(f"  Overall missing %:     {(total_missing / total_cells) * 100:.4f}%")
    print(f"  Rows with any missing: {df.isnull().any(axis=1).sum():,}")
    print(f"  Complete rows:         {df.dropna().shape[0]:,}")
    
    cols_with_missing = missing_stats[missing_stats['Missing Count'] > 0]
    
    if len(cols_with_missing) > 0:
        print(f"\n" + "-" * 60)
        print("Columns with Missing Values:")
        print("-" * 60)
        display(cols_with_missing) if 'display' in dir() else print(cols_with_missing)
    else:
        print("\n✓ No missing values found!")
    
    # Hidden missing values
    print("\n" + "-" * 60)
    print("Hidden Missing Values Check:")
    print("-" * 60)
    
    hidden_patterns = ['', ' ', 'nan', 'null', 'none', 'n/a', 'na', '-', '.']
    
    for col in df.columns:
        col_str = df[col].astype(str).str.lower().str.strip()
        hidden_count = col_str.isin(hidden_patterns).sum()
        if hidden_count > 0:
            print(f"  {col:<25} {hidden_count:>8,} hidden missing values")
    
    return missing_stats


missing_analysis = analyze_missing_values(df_raw)

MISSING VALUES ANALYSIS

Overall Statistics:
  Total cells:           4,335,272
  Total missing:         109,025
  Overall missing %:     2.5148%
  Rows with any missing: 108,221
  Complete rows:         433,688

------------------------------------------------------------
Columns with Missing Values:
------------------------------------------------------------
             Missing Count  Missing %  Present Count           Dtype
CustomerID          108000      19.93         433909  string[python]
Description           1025       0.19         540884  string[python]

------------------------------------------------------------
Hidden Missing Values Check:
------------------------------------------------------------


## 6. Noise Character Analysis

In [89]:
# =============================================================================
# CELL 6: NOISE CHARACTER ANALYSIS
# =============================================================================

NOISE_PATTERNS = {
    'ö (umlaut)': r'ö',
    'ä (umlaut)': r'ä',
    'ü (umlaut)': r'ü',
    '^': r'\^',
    '&': r'&',
    '#': r'#',
    '@': r'@',
    '$': r'\$',
    '☺️ (emoji)': r'☺️?',
    'XxY': r'XxY',
    'Ww': r'Ww',
    'Non-ASCII': r'[^\x00-\x7F]',
}


def analyze_noise_in_column(df: pd.DataFrame, column: str, show_examples: int = 3) -> dict:
    """Analyze noise characters in a specific column."""
    if column not in df.columns:
        return {}
    
    col_str = df[column].astype(str)
    results = {}
    
    print(f"\n{'='*60}")
    print(f"NOISE ANALYSIS: {column.upper()}")
    print(f"{'='*60}")
    
    total_rows = len(df)
    rows_with_any_noise = set()
    
    for name, pattern in NOISE_PATTERNS.items():
        try:
            matches_mask = col_str.str.contains(pattern, na=False, regex=True)
            match_count = matches_mask.sum()
            
            if match_count > 0:
                rows_with_any_noise.update(df[matches_mask].index.tolist())
                match_pct = (match_count / total_rows) * 100
                
                print(f"\n  '{name}' found in {match_count:,} rows ({match_pct:.2f}%)")
                
                examples = df[matches_mask][column].head(show_examples).tolist()
                for i, ex in enumerate(examples, 1):
                    ex_display = str(ex)[:70] + "..." if len(str(ex)) > 70 else str(ex)
                    print(f"    Example {i}: {ex_display}")
                
                results[name] = {'count': match_count, 'percentage': match_pct}
        except Exception as e:
            print(f"  Warning: Could not check pattern '{name}': {e}")
    
    print(f"\n{'-'*60}")
    print(f"SUMMARY: {len(rows_with_any_noise):,} rows with any noise ({len(rows_with_any_noise)/total_rows*100:.2f}%)")
    
    return results


# Analyze key columns
columns_to_check = ['Description', 'StockCode', 'Country', 'CustomerID', 'Quantity', 'UnitPrice']
for col in columns_to_check:
    if col in df_raw.columns:
        analyze_noise_in_column(df_raw, col)


NOISE ANALYSIS: DESCRIPTION

  '&' found in 2,718 rows (0.50%)
    Example 1: $CHARLIE & LOLA WASTEPAPER BIN FLORA
    Example 2: VINTAGE SNAKES & LADDERS
    Example 3: $LADIES & GENTLEMEN METAL SIGN

  '$' found in 162,466 rows (29.98%)
    Example 1: $KNITTED UNION FLAG HOT WATER BOTTLE
    Example 2: $RED WOOLLY HOTTIE WHITE HEART.
    Example 3: $GLASS STAR FROSTED T-LIGHT HOLDER

  'Non-ASCII' found in 32 rows (0.01%)
    Example 1: $Dotcomgiftshop Gift Voucher £40.00
    Example 2: $Dotcomgiftshop Gift Voucher £50.00
    Example 3: Dotcomgiftshop Gift Voucher £30.00

------------------------------------------------------------
SUMMARY: 164,372 rows with any noise (30.33%)

NOISE ANALYSIS: STOCKCODE

  'ö (umlaut)' found in 271,235 rows (50.05%)
    Example 1: ö84406B^
    Example 2: ö21730^
    Example 3: ö22632^

  '^' found in 271,235 rows (50.05%)
    Example 1: ö84406B^
    Example 2: ö21730^
    Example 3: ö22632^

  'Non-ASCII' found in 271,235 rows (50.05%)
    Example 1

## 7. Duplicate Analysis

In [90]:
# =============================================================================
# CELL 7: DUPLICATE ANALYSIS
# =============================================================================

def analyze_duplicates(df: pd.DataFrame, show_examples: int = 5) -> dict:
    """Comprehensive duplicate analysis."""
    print("=" * 60)
    print("DUPLICATE ANALYSIS")
    print("=" * 60)
    
    results = {}
    total_rows = len(df)
    
    # Full row duplicates
    full_dupes_count = df.duplicated(keep='first').sum()
    
    print(f"\n1. EXACT ROW DUPLICATES")
    print("-" * 40)
    print(f"   Duplicate rows:      {full_dupes_count:,}")
    print(f"   Percentage:          {(full_dupes_count/total_rows)*100:.4f}%")
    
    results['exact_duplicates'] = full_dupes_count
    
    # Key column combinations
    key_columns = [
        ['StockCode'],
        ['Description'],
        ['StockCode', 'Description'],
    ]
    
    print(f"\n2. DUPLICATES BY KEY COLUMNS")
    print("-" * 40)
    
    for keys in key_columns:
        if all(k in df.columns for k in keys):
            key_str = " + ".join(keys)
            dupe_count = df.duplicated(subset=keys, keep='first').sum()
            unique_count = df[keys].drop_duplicates().shape[0]
            
            print(f"\n   {key_str}:")
            print(f"     Duplicate rows:      {dupe_count:,}")
            print(f"     Unique combinations: {unique_count:,}")
            
            results[key_str] = {'duplicates': dupe_count, 'unique': unique_count}
    
    return results


duplicate_results = analyze_duplicates(df_raw)

DUPLICATE ANALYSIS

1. EXACT ROW DUPLICATES
----------------------------------------
   Duplicate rows:      91
   Percentage:          0.0168%

2. DUPLICATES BY KEY COLUMNS
----------------------------------------

   StockCode:
     Duplicate rows:      534,104
     Unique combinations: 7,805

   Description:
     Duplicate rows:      533,952
     Unique combinations: 7,957

   StockCode + Description:
     Duplicate rows:      524,922
     Unique combinations: 16,987


## 8. Data Cleaning

In [91]:
# =============================================================================
# CELL 8: DATA CLEANING
# =============================================================================

class DataCleaner:
    """Robust data cleaning class for e-commerce product data."""
    
    NOISE_PATTERNS = [
        r'ö', r'ä', r'ü', r'\^', r'☺️?', r'[\U0001F600-\U0001F64F]',
        r'XxY', r'Ww', r'@', r'\$', r'#', r'&', r'[\x00-\x1F\x7F]',
    ]
    
    COMBINED_NOISE_PATTERN = re.compile('|'.join(NOISE_PATTERNS))
    WHITESPACE_PATTERN = re.compile(r'\s+')
    
    def __init__(self, df: pd.DataFrame):
        self.df_original = df
        self.df = df.copy()
        self.cleaning_log = []
        self._log(f"Initialized cleaner with {len(df):,} rows")
    
    def _log(self, message: str) -> None:
        self.cleaning_log.append(message)
        print(f"  → {message}")
    
    @staticmethod
    def clean_text(text: str) -> Optional[str]:
        if pd.isna(text) or not isinstance(text, str):
            return text
        cleaned = DataCleaner.COMBINED_NOISE_PATTERN.sub('', text)
        cleaned = DataCleaner.WHITESPACE_PATTERN.sub(' ', cleaned)
        return cleaned.strip()
    
    @staticmethod
    def clean_stock_code(code) -> Optional[str]:
        if pd.isna(code):
            return None
        cleaned = DataCleaner.clean_text(str(code))
        if cleaned is None:
            return None
        cleaned = re.sub(r'[^a-zA-Z0-9-]', '', cleaned)
        return cleaned if cleaned else None
    
    @staticmethod
    def clean_country(country) -> Optional[str]:
        if pd.isna(country):
            return None
        cleaned = DataCleaner.clean_text(str(country))
        if cleaned is None:
            return None
        cleaned = re.sub(r'[^a-zA-Z\s]', '', cleaned).strip()
        return cleaned if cleaned else None
    
    @staticmethod
    def clean_quantity(qty) -> int:
        if pd.isna(qty):
            return 0
        cleaned = DataCleaner.COMBINED_NOISE_PATTERN.sub('', str(qty))
        match = re.search(r'^-?\d+\.?\d*', cleaned.strip())
        if match:
            try:
                return int(float(match.group()))
            except (ValueError, OverflowError):
                return 0
        return 0
    
    @staticmethod
    def clean_price(price) -> Optional[float]:
        if pd.isna(price):
            return None
        cleaned = DataCleaner.COMBINED_NOISE_PATTERN.sub('', str(price))
        cleaned = re.sub(r'[£€$,]', '', cleaned)
        match = re.search(r'-?\d+\.?\d*', cleaned.strip())
        if match:
            try:
                return float(match.group())
            except (ValueError, OverflowError):
                return None
        return None
    
    @staticmethod
    def clean_customer_id(cid) -> Optional[str]:
        if pd.isna(cid):
            return None
        cleaned = DataCleaner.COMBINED_NOISE_PATTERN.sub('', str(cid))
        match = re.search(r'\d+\.?\d*', cleaned)
        if match:
            num_str = match.group()
            if '.' in num_str:
                try:
                    num = float(num_str)
                    if num == int(num):
                        return str(int(num))
                    return num_str
                except ValueError:
                    return None
            return num_str
        return None
    
    def clean_all_columns(self) -> 'DataCleaner':
        print("\n" + "=" * 60)
        print("CLEANING ALL COLUMNS")
        print("=" * 60)
        
        if 'Description' in self.df.columns:
            self._log("Cleaning Description...")
            self.df['Description'] = self.df['Description'].apply(self.clean_text)
        
        if 'StockCode' in self.df.columns:
            self._log("Cleaning StockCode...")
            self.df['StockCode'] = self.df['StockCode'].apply(self.clean_stock_code)
        
        if 'Country' in self.df.columns:
            self._log("Cleaning Country...")
            self.df['Country'] = self.df['Country'].apply(self.clean_country)
        
        if 'Quantity' in self.df.columns:
            self._log("Cleaning Quantity...")
            self.df['Quantity'] = self.df['Quantity'].apply(self.clean_quantity)
        
        if 'UnitPrice' in self.df.columns:
            self._log("Cleaning UnitPrice...")
            self.df['UnitPrice'] = self.df['UnitPrice'].apply(self.clean_price)
        
        if 'CustomerID' in self.df.columns:
            self._log("Cleaning CustomerID...")
            self.df['CustomerID'] = self.df['CustomerID'].apply(self.clean_customer_id)
        
        if 'InvoiceDate' in self.df.columns:
            self._log("Converting InvoiceDate to datetime...")
            self.df['InvoiceDate'] = pd.to_datetime(self.df['InvoiceDate'], errors='coerce')
        
        return self
    
    def show_comparison(self, n_rows: int = 5) -> None:
        print("\n" + "=" * 60)
        print("BEFORE/AFTER COMPARISON")
        print("=" * 60)
        
        for col in ['StockCode', 'Description', 'Country', 'Quantity', 'UnitPrice']:
            if col in self.df.columns and col in self.df_original.columns:
                original = self.df_original[col].astype(str)
                cleaned = self.df[col].astype(str)
                diff_mask = original != cleaned
                diff_count = diff_mask.sum()
                
                if diff_count > 0:
                    print(f"\n{col} ({diff_count:,} rows changed):")
                    comparison = pd.DataFrame({
                        'Before': self.df_original.loc[diff_mask, col].head(n_rows),
                        'After': self.df.loc[diff_mask, col].head(n_rows)
                    })
                    display(comparison) if 'display' in dir() else print(comparison)
    
    def get_cleaned_df(self) -> pd.DataFrame:
        return self.df


# Apply cleaning
cleaner = DataCleaner(df_raw)
cleaner.clean_all_columns()
cleaner.show_comparison(n_rows=5)
df_clean = cleaner.get_cleaned_df()

print("\n" + "=" * 60)
print("CLEANING COMPLETE")
print("=" * 60)
print(f"Original rows: {len(df_raw):,}")
print(f"Cleaned rows:  {len(df_clean):,}")

  → Initialized cleaner with 541,909 rows

CLEANING ALL COLUMNS
  → Cleaning Description...
  → Cleaning StockCode...
  → Cleaning Country...
  → Cleaning Quantity...
  → Cleaning UnitPrice...
  → Cleaning CustomerID...
  → Converting InvoiceDate to datetime...

BEFORE/AFTER COMPARISON

StockCode (271,276 rows changed):
      Before   After
2   ö84406B^  84406B
6    ö21730^   21730
8    ö22632^   22632
9    ö84879^   84879
10   ö22745^   22745

Description (254,018 rows changed):
                                  Before                                After
3   $KNITTED UNION FLAG HOT WATER BOTTLE  KNITTED UNION FLAG HOT WATER BOTTLE
4        $RED WOOLLY HOTTIE WHITE HEART.       RED WOOLLY HOTTIE WHITE HEART.
6     $GLASS STAR FROSTED T-LIGHT HOLDER    GLASS STAR FROSTED T-LIGHT HOLDER
9         $ASSORTED COLOUR BIRD ORNAMENT        ASSORTED COLOUR BIRD ORNAMENT
10           $POPPY'S PLAYHOUSE BEDROOM             POPPY'S PLAYHOUSE BEDROOM

Country (270,373 rows changed):
              

## Cell 9 - Handle Missing Values

In [92]:
# =============================================================================
# CELL 9: HANDLE MISSING VALUES
# =============================================================================

def handle_missing_values(
    df: pd.DataFrame,
    critical_columns: list = ['Description', 'StockCode'],
    fill_strategies: dict = None
) -> pd.DataFrame:
    """Handle missing values with configurable strategies."""
    print("=" * 60)
    print("HANDLING MISSING VALUES")
    print("=" * 60)
    
    df_result = df.copy()
    initial_rows = len(df_result)
    
    print(f"\nInitial rows: {initial_rows:,}")
    
    missing_cols = df_result.isnull().sum()
    missing_cols = missing_cols[missing_cols > 0]
    
    if len(missing_cols) > 0:
        print(f"\nColumns with missing values:")
        for col, count in missing_cols.items():
            print(f"  {col:<20} {count:>8,} ({count/initial_rows*100:.2f}%)")
    
    # Drop rows with missing critical columns
    print(f"\n" + "-" * 40)
    print("Dropping rows with missing critical columns:")
    
    for col in critical_columns:
        if col in df_result.columns:
            missing_count = df_result[col].isnull().sum()
            if missing_count > 0:
                df_result = df_result.dropna(subset=[col])
                print(f"  Dropped {missing_count:,} rows with missing '{col}'")
    
    # Apply fill strategies
    if fill_strategies:
        print(f"\nApplying fill strategies:")
        for col, strategy in fill_strategies.items():
            if col in df_result.columns:
                missing_before = df_result[col].isnull().sum()
                if missing_before > 0:
                    if callable(strategy):
                        df_result[col] = df_result[col].fillna(strategy(df_result[col]))
                    else:
                        df_result[col] = df_result[col].fillna(strategy)
                    print(f"  Filled {missing_before:,} missing '{col}'")
    
    print(f"\n" + "=" * 60)
    print(f"Rows: {initial_rows:,} → {len(df_result):,} (dropped {initial_rows - len(df_result):,})")
    
    return df_result


df_clean = handle_missing_values(
    df_clean,
    critical_columns=['Description', 'StockCode'],
    fill_strategies={'Quantity': 0, 'UnitPrice': lambda x: x.median()}
)

HANDLING MISSING VALUES

Initial rows: 541,909

Columns with missing values:
  Description             1,025 (0.19%)
  CustomerID            135,080 (24.93%)

----------------------------------------
Dropping rows with missing critical columns:
  Dropped 1,025 rows with missing 'Description'

Applying fill strategies:

Rows: 541,909 → 540,884 (dropped 1,025)


## 10. Create Products DataFrame

In [93]:
# =============================================================================
# CELL 10: CREATE PRODUCTS DATAFRAME
# =============================================================================

def create_products_dataframe(
    df: pd.DataFrame,
    group_columns: list = ['StockCode', 'Description']
) -> pd.DataFrame:
    """Create a unique products DataFrame from transaction data."""
    print("=" * 60)
    print("CREATING PRODUCTS DATAFRAME")
    print("=" * 60)
    
    df_result = df.copy()
    initial_rows = len(df_result)
    
    # Ensure UnitPrice is numeric
    if 'UnitPrice' in df_result.columns:
        if df_result['UnitPrice'].dtype == 'object':
            df_result['UnitPrice'] = pd.to_numeric(df_result['UnitPrice'], errors='coerce')
        
        price_nulls = df_result['UnitPrice'].isnull().sum()
        if price_nulls > 0:
            df_result = df_result.dropna(subset=['UnitPrice'])
            print(f"  Dropped {price_nulls:,} rows with invalid UnitPrice")
    
    # Remove exact duplicates
    dupes_before = df_result.duplicated().sum()
    df_result = df_result.drop_duplicates()
    print(f"  Removed {dupes_before:,} exact duplicate rows")
    print(f"  Rows after deduplication: {len(df_result):,}")
    
    # Aggregation config
    agg_config = {
        'UnitPrice': ['mean', 'min', 'max', 'std'],
        'Quantity': 'sum',
        'Country': lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else x.iloc[0],
        'InvoiceNo': 'count'
    }
    
    agg_config_filtered = {k: v for k, v in agg_config.items() if k in df_result.columns}
    
    print(f"\nGrouping by {group_columns}...")
    df_products = df_result.groupby(group_columns, as_index=False).agg(agg_config_filtered)
    
    # Flatten column names
    if isinstance(df_products.columns, pd.MultiIndex):
        df_products.columns = ['_'.join(col).strip('_') for col in df_products.columns]
    
    # Rename columns
    column_renames = {
        'UnitPrice_mean': 'UnitPrice',
        'UnitPrice_min': 'PriceMin',
        'UnitPrice_max': 'PriceMax',
        'UnitPrice_std': 'PriceStd',
        'Quantity_sum': 'TotalQuantitySold',
        'Country_<lambda>': 'PrimaryCountry',
        'InvoiceNo_count': 'TransactionCount'
    }
    
    df_products = df_products.rename(columns={k: v for k, v in column_renames.items() if k in df_products.columns})
    
    print(f"\n" + "=" * 60)
    print(f"Original transaction rows: {initial_rows:,}")
    print(f"Unique products:           {len(df_products):,}")
    
    return df_products


df_products = create_products_dataframe(df_clean)
display(df_products.head(10)) if 'display' in dir() else print(df_products.head(10))

CREATING PRODUCTS DATAFRAME
  Removed 2,732 exact duplicate rows
  Rows after deduplication: 538,152

Grouping by ['StockCode', 'Description']...

Original transaction rows: 540,884
Unique products:           5,163
  StockCode                  Description  UnitPrice  PriceMin  PriceMax  \
0     10002   INFLATABLE POLITICAL GLOBE       1.09      0.85      1.66   
1     10002                          nan       0.00      0.00      0.00   
2     10080     GROOVY CACTUS INFLATABLE       0.41      0.39      0.85   
3     10080                        check       0.00      0.00      0.00   
4     10120                 DOGGY RUBBER       0.21      0.21      0.21   
5    10123C         HEARTS WRAPPING TAPE       0.65      0.65      0.65   
6    10123C                          nan       0.00      0.00      0.00   
7    10124A  SPOTS ON RED BOOKCOVER TAPE       0.42      0.42      0.42   
8    10124G     ARMY CAMO BOOKCOVER TAPE       0.42      0.42      0.42   
9     10125      MINI FUNKY DESIGN 

## Cell 11 - Standardize & Filter Products

In [94]:
# =============================================================================
# CELL 11: STANDARDIZE & FILTER PRODUCTS
# =============================================================================

def standardize_products(
    df: pd.DataFrame,
    min_price: float = 0.01,
    min_description_length: int = 3
) -> pd.DataFrame:
    """Standardize and filter the products DataFrame."""
    print("=" * 60)
    print("STANDARDIZING & FILTERING PRODUCTS")
    print("=" * 60)
    
    df_result = df.copy()
    initial_count = len(df_result)
    
    # Standardize text
    print("\n1. STANDARDIZING TEXT")
    print("-" * 40)
    
    if 'Description' in df_result.columns:
        df_result['Description'] = df_result['Description'].astype(str).str.upper().str.strip()
        print("  ✓ Description: uppercase + stripped")
    
    if 'PrimaryCountry' in df_result.columns:
        df_result['PrimaryCountry'] = df_result['PrimaryCountry'].astype(str).str.strip().str.title()
        print("  ✓ PrimaryCountry: stripped + title case")
    
    if 'StockCode' in df_result.columns:
        df_result['StockCode'] = df_result['StockCode'].astype(str).str.strip().str.upper()
        print("  ✓ StockCode: stripped + uppercase")
    
    if 'UnitPrice' in df_result.columns:
        df_result['UnitPrice'] = df_result['UnitPrice'].round(2)
        print("  ✓ UnitPrice: rounded to 2 decimals")
    
    # Apply filters
    print("\n2. APPLYING FILTERS")
    print("-" * 40)
    
    # Price filter
    if 'UnitPrice' in df_result.columns:
        before = len(df_result)
        df_result = df_result[df_result['UnitPrice'] > min_price]
        print(f"  Removed {before - len(df_result):,} products with price <= ${min_price}")
    
    # Description length filter
    if 'Description' in df_result.columns:
        before = len(df_result)
        df_result = df_result[df_result['Description'].str.len() > min_description_length]
        print(f"  Removed {before - len(df_result):,} products with description <= {min_description_length} chars")
    
    # Exclude patterns
    exclude_patterns = [
        'DAMAGED', 'LOST IN', 'WRONG', 'CRUSHED', 'FOUND', 'AMAZON FEE', 
        'MANUAL', 'BANK CHARGES', 'POSTAGE', 'DOTCOM', 'ADJUST', 'SAMPLES', 
        'CRUK', 'COMMISSION', 'DISCOUNT', 'TEST', 'CARRIAGE', 'MAILOUT',
        'AMAZON', 'EBAY', 'RETURNED', 'MISSING', 'BREAKAGE', 'THROWN',
        r'\?+',
    ]
    
    if 'Description' in df_result.columns:
        safe_patterns = [re.escape(p) if not any(c in p for c in [r'\?', r'\+', r'\*']) else p for p in exclude_patterns]
        combined_pattern = '|'.join(safe_patterns)
        
        before = len(df_result)
        mask = ~df_result['Description'].str.contains(combined_pattern, case=False, na=False, regex=True)
        df_result = df_result[mask]
        print(f"  Removed {before - len(df_result):,} products matching exclude patterns")
    
    # Remove invalid entries
    print("\n3. FINAL CLEANUP")
    print("-" * 40)
    
    before = len(df_result)
    df_result = df_result.dropna(subset=['StockCode', 'Description', 'UnitPrice'])
    df_result = df_result[df_result['Description'].str.strip() != '']
    df_result = df_result[~df_result['Description'].str.upper().isin(['NAN', 'NONE', 'NULL'])]
    print(f"  Removed {before - len(df_result):,} invalid entries")
    
    print(f"\n" + "=" * 60)
    print(f"Initial: {initial_count:,} → Final: {len(df_result):,}")
    
    return df_result


df_products = standardize_products(df_products)
display(df_products.head(10)) if 'display' in dir() else print(df_products.head(10))

STANDARDIZING & FILTERING PRODUCTS

1. STANDARDIZING TEXT
----------------------------------------
  ✓ Description: uppercase + stripped
  ✓ PrimaryCountry: stripped + title case
  ✓ StockCode: stripped + uppercase
  ✓ UnitPrice: rounded to 2 decimals

2. APPLYING FILTERS
----------------------------------------
  Removed 997 products with price <= $0.01
  Removed 0 products with description <= 3 chars
  Removed 21 products matching exclude patterns

3. FINAL CLEANUP
----------------------------------------
  Removed 0 invalid entries

Initial: 5,163 → Final: 4,145
   StockCode                   Description  UnitPrice  PriceMin  PriceMax  \
0      10002    INFLATABLE POLITICAL GLOBE       1.09      0.85      1.66   
2      10080      GROOVY CACTUS INFLATABLE       0.41      0.39      0.85   
4      10120                  DOGGY RUBBER       0.21      0.21      0.21   
5     10123C          HEARTS WRAPPING TAPE       0.65      0.65      0.65   
7     10124A   SPOTS ON RED BOOKCOVER TAPE 

## 12. Final Noise Cleanup

In [95]:
# =============================================================================
# CELL 12: FINAL NOISE CLEANUP
# =============================================================================

def final_noise_cleanup(df: pd.DataFrame) -> pd.DataFrame:
    """Remove ALL non-ASCII characters (aggressive final pass)."""
    print("=" * 60)
    print("FINAL NOISE CLEANUP")
    print("=" * 60)
    
    df_result = df.copy()
    
    def aggressive_clean(text):
        if pd.isna(text):
            return text
        text = str(text)
        text = re.sub(r'XxY', '', text, flags=re.IGNORECASE)
        text = re.sub(r'Ww', '', text, flags=re.IGNORECASE)
        text = ''.join(char for char in text if 32 <= ord(char) <= 126)
        text = re.sub(r'\s+', ' ', text).strip()
        return text
    
    text_columns = ['Description', 'StockCode', 'PrimaryCountry']
    
    for col in text_columns:
        if col in df_result.columns:
            before = df_result[col].astype(str).str.contains(r'[^\x00-\x7F]|XxY|Ww', regex=True, na=False).sum()
            df_result[col] = df_result[col].apply(aggressive_clean)
            after = df_result[col].astype(str).str.contains(r'[^\x00-\x7F]|XxY|Ww', regex=True, na=False).sum()
            print(f"  {col}: {before} → {after} noisy rows")
    
    # Remove empty entries
    before = len(df_result)
    df_result = df_result[df_result['Description'].str.strip() != '']
    df_result = df_result[df_result['Description'].str.len() >= 3]
    df_result = df_result[df_result['StockCode'].str.strip() != '']
    print(f"  Removed {before - len(df_result)} invalid entries")
    
    # Verify
    total_noise = sum(
        df_result[col].astype(str).str.contains(r'[^\x00-\x7F]|XxY|Ww', regex=True, na=False).sum()
        for col in text_columns if col in df_result.columns
    )
    
    print(f"\n  TOTAL NOISE: {total_noise} {'✓ CLEAN!' if total_noise == 0 else '⚠'}")
    print(f"  Final rows: {len(df_result):,}")
    
    return df_result


df_products = final_noise_cleanup(df_products)

FINAL NOISE CLEANUP
  Description: 0 → 0 noisy rows
  StockCode: 0 → 0 noisy rows
  PrimaryCountry: 0 → 0 noisy rows
  Removed 0 invalid entries

  TOTAL NOISE: 0 ✓ CLEAN!
  Final rows: 4,145
  Description: 0 → 0 noisy rows
  StockCode: 0 → 0 noisy rows
  PrimaryCountry: 0 → 0 noisy rows
  Removed 0 invalid entries

  TOTAL NOISE: 0 ✓ CLEAN!
  Final rows: 4,145


## 13. Handle Nulls & Save Cleaned Data

In [96]:
# =============================================================================
# CELL 13: HANDLE NULLS, SAVE & VALIDATE
# =============================================================================

def handle_remaining_nulls(df: pd.DataFrame) -> pd.DataFrame:
    """Handle any remaining null values before saving."""
    print("=" * 60)
    print("HANDLING REMAINING NULL VALUES")
    print("=" * 60)
    
    df_result = df.copy()
    
    # Check which columns have nulls
    null_summary = df_result.isnull().sum()
    null_cols = null_summary[null_summary > 0]
    
    if len(null_cols) == 0:
        print("  ✓ No null values found!")
        return df_result
    
    print(f"\n  Columns with nulls:")
    for col, count in null_cols.items():
        print(f"    {col}: {count}")
    
    # Fill / drop nulls based on column type
    for col in null_cols.index:
        if df_result[col].dtype == 'object':
            df_result[col] = df_result[col].fillna("Unknown")
            print(f"  → Filled nulls in '{col}' with 'Unknown'")
        elif df_result[col].dtype in ['int64', 'float64']:
            # Fill numeric columns with median
            median_val = df_result[col].median()
            df_result[col] = df_result[col].fillna(median_val)
            print(f"  → Filled nulls in '{col}' with median: {median_val}")
        else:
            # For other types, drop rows
            before = len(df_result)
            df_result = df_result.dropna(subset=[col])
            after = len(df_result)
            print(f"  → Dropped {before - after} rows due to nulls in '{col}'")
    
    print("\n  ✓ Null handling complete")
    return df_result


def save_and_validate(df: pd.DataFrame, output_path: Path) -> dict:
    """Save cleaned DataFrame and validate the saved file."""
    print("=" * 60)
    print("SAVING CLEANED DATA")
    print("=" * 60)
    
    # Handle remaining nulls
    df = handle_remaining_nulls(df)
    
    # Create directory
    output_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Save
    df.to_csv(output_path, index=False, encoding='utf-8')
    file_size_mb = output_path.stat().st_size / (1024 * 1024)
    
    print(f"  ✓ Saved {len(df):,} products")
    print(f"  ✓ File: {output_path}")
    print(f"  ✓ Size: {file_size_mb:.2f} MB")
    
    # Validate
    print("\n" + "=" * 60)
    print("VALIDATING SAVED FILE")
    print("=" * 60)
    
    df_verify = pd.read_csv(output_path)
    
    text_columns = ['Description', 'StockCode', 'PrimaryCountry']
    noise_count = sum(
        df_verify[col].astype(str).str.contains(r'[^\x00-\x7F]|XxY|Ww', regex=True, na=False).sum()
        for col in text_columns if col in df_verify.columns
    )
    
    null_count = df_verify.isnull().sum().sum()
    empty_desc = (df_verify['Description'].str.strip() == '').sum() if 'Description' in df_verify.columns else 0
    invalid_prices = (df_verify['UnitPrice'] <= 0).sum() if 'UnitPrice' in df_verify.columns else 0
    dupes = df_verify.duplicated().sum()
    
    results = {
        'null_values': null_count,
        'empty_descriptions': empty_desc,
        'invalid_prices': invalid_prices,
        'duplicates': dupes,
        'noise_characters': noise_count
    }
    
    print(f"  Null values:       {null_count} {'✓' if null_count == 0 else '⚠'}")
    print(f"  Empty descriptions: {empty_desc} {'✓' if empty_desc == 0 else '⚠'}")
    print(f"  Invalid prices:    {invalid_prices} {'✓' if invalid_prices == 0 else '⚠'}")
    print(f"  Duplicate rows:    {dupes} {'✓' if dupes == 0 else '⚠'}")
    print(f"  Noise characters:  {noise_count} {'✓' if noise_count == 0 else '⚠'}")
    
    all_passed = all(v == 0 for v in results.values())
    
    print("\n" + "=" * 60)
    if all_passed:
        print("✓ ALL CHECKS PASSED!")
        print(f"✓ Ready for 02_embedding_generation.ipynb")
        print(f"✓ Products to embed: {len(df_verify):,}")
    else:
        print("⚠ SOME CHECKS FAILED - Review above")
    print("=" * 60)
    
    return results


# Example usage
cleaned_data_path = backend_path / 'data' / 'cleaned' / 'products_cleaned.csv'
validation_results = save_and_validate(df_products, cleaned_data_path)


SAVING CLEANED DATA
HANDLING REMAINING NULL VALUES

  Columns with nulls:
    PriceStd: 197
  → Filled nulls in 'PriceStd' with median: 0.6125301160598985

  ✓ Null handling complete
  ✓ Saved 4,145 products
  ✓ File: /home/chris/coding-task/productlens-ai/backend/data/cleaned/products_cleaned.csv
  ✓ Size: 0.35 MB

VALIDATING SAVED FILE
  Null values:       0 ✓
  Empty descriptions: 0 ✓
  Invalid prices:    0 ✓
  Duplicate rows:    0 ✓
  Noise characters:  0 ✓

✓ ALL CHECKS PASSED!
✓ Ready for 02_embedding_generation.ipynb
✓ Products to embed: 4,145
  Null values:       0 ✓
  Empty descriptions: 0 ✓
  Invalid prices:    0 ✓
  Duplicate rows:    0 ✓
  Noise characters:  0 ✓

✓ ALL CHECKS PASSED!
✓ Ready for 02_embedding_generation.ipynb
✓ Products to embed: 4,145


## 14. Final Summary

In [97]:
# =============================================================================
# CELL 14: FINAL SUMMARY
# =============================================================================

print("=" * 60)
print("DATA CLEANING NOTEBOOK COMPLETE")
print("=" * 60)

print(f"""
SUMMARY:
  • Original rows:     {len(df_raw):,}
  • Final products:    {len(df_products):,}
  • Reduction:         {(1 - len(df_products)/len(df_raw))*100:.2f}%
  • Output file:       {cleaned_data_path}

COLUMNS IN CLEANED DATA:
  {df_products.columns.tolist()}

NEXT STEPS:
  1. Open 02_embedding_generation.ipynb
  2. Generate embeddings for {len(df_products):,} products
  3. Upload to Pinecone vector database
""")

DATA CLEANING NOTEBOOK COMPLETE

SUMMARY:
  • Original rows:     541,909
  • Final products:    4,145
  • Reduction:         99.24%
  • Output file:       /home/chris/coding-task/productlens-ai/backend/data/cleaned/products_cleaned.csv

COLUMNS IN CLEANED DATA:
  ['StockCode', 'Description', 'UnitPrice', 'PriceMin', 'PriceMax', 'PriceStd', 'TotalQuantitySold', 'PrimaryCountry', 'TransactionCount']

NEXT STEPS:
  1. Open 02_embedding_generation.ipynb
  2. Generate embeddings for 4,145 products
  3. Upload to Pinecone vector database

